# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Joint and Conditional Random Variables

This notebook accompanies Chapter 2, Sections 2.6.1--2.6.3, 2.6.5, and
2.7. We compute marginals, dependence, covariance, conditional laws,
conditional expectations, and the effect of a fixed filter.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(303)


## Start with a finite joint distribution

Let $X,Y\in\{0,1\}$. Rows below correspond to $X=0,1$, columns to
$Y=0,1$, and the entries specify the complete joint PMF:

$$
p_{X,Y}=\begin{pmatrix}0.30&0.20\\0.10&0.40\end{pmatrix}.
$$

Marginal probabilities are obtained by summing rows or columns. A
conditional PMF is obtained by dividing a joint probability by a positive
marginal probability.


In [ ]:
joint = np.array([[0.30, 0.20],
                  [0.10, 0.40]])
if np.any(joint < 0) or not np.isclose(joint.sum(), 1):
    raise ValueError("A joint PMF must be nonnegative and sum to one.")

p_x = joint.sum(axis=1)
p_y = joint.sum(axis=0)
p_y_given_x = joint / p_x[:, None]

print("P(X=x):", p_x)
print("P(Y=y):", p_y)
print("rows of P(Y=y | X=x):\n", p_y_given_x)
print("Independent?", np.allclose(joint, p_x[:, None] * p_y[None, :]))


The conditional mean is a function of the conditioning value. Here

$$
m(x)=\mathbb E[Y\mid X=x]
=\sum_y y\,p_{Y\mid X}(y\mid x).
$$

The tower property gives
$\mathbb E[\mathbb E[Y\mid X]]=\mathbb E[Y]$. Covariance is
$\operatorname{Cov}(X,Y)=\mathbb E[XY]-\mathbb E[X]\mathbb E[Y]$.
Zero covariance alone does not generally imply independence.


In [ ]:
values = np.array([0, 1])
conditional_mean_y = p_y_given_x @ values
e_y_by_tower = np.sum(conditional_mean_y * p_x)
e_y = np.sum(values * p_y)
e_x = np.sum(values * p_x)
e_xy = sum(x * y * joint[x, y] for x in values for y in values)
cov_xy = e_xy - e_x * e_y

print("E[Y | X=0], E[Y | X=1] =", conditional_mean_y)
print("E[E[Y | X]] =", e_y_by_tower, "and E[Y] =", e_y)
print("Cov(X,Y) =", cov_xy)


## A continuous example

Consider

$$
f_{X,Y}(x,y)=2\mathbf 1_{\{0<x<y<1\}}.
$$

Integrating over the triangular support gives

$$
f_X(x)=2(1-x),\qquad f_Y(y)=2y,\qquad 0<x,y<1.
$$

For $0<y<1$, $f_Y(y)>0$, and
$f_{X\mid Y}(x\mid y)=1/y$ on $0<x<y$. Thus
$\mathbb E[X\mid Y=y]=y/2$. Conditioning on the value of a continuous
random variable uses conditional densities; it is not the ratio of two
positive event probabilities because $\mathbb P(Y=y)=0$.


In [ ]:
n = 40_000
# First sample Y with density 2y, then X | Y=y uniformly on (0,y).
y_cont = np.sqrt(rng.random(n))
x_cont = y_cont * rng.random(n)

print(f"Simulated E[X] = {x_cont.mean():.4f}; theory = {1/3:.4f}")
print(f"Simulated E[Y] = {y_cont.mean():.4f}; theory = {2/3:.4f}")
print(f"Simulated Cov(X,Y) = {np.cov(x_cont, y_cont, ddof=0)[0, 1]:.4f}; theory = {1/36:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(x_cont[:2500], y_cont[:2500], s=7, alpha=0.25)
ax.plot([0, 1], [0, 1], color="black", linestyle="--")
ax.set(xlabel="X", ylabel="Y", title=r"Joint support $0<X<Y<1$")
plt.show()


In [ ]:
# Check the tower property numerically: E[E[X | Y]] = E[Y/2] = E[X].
print("mean of Y/2:", (y_cont / 2).mean())
print("mean of X:  ", x_cont.mean())


## Keeping only selected observations

Suppose $X_1,\ldots,X_n$ are independent fair die rolls. Fix the filter
$A=\{5,6\}$ **before** seeing the data and retain the rolls in $A$.
The retained count $N_A$ is random. Conditional on $N_A=k\geq1$, the
retained values are i.i.d. from the original law conditional on $A$. In
particular, half of that conditional law is at 6. A ratio based on retained
values is undefined when $N_A=0$; code must handle this case explicitly.


In [ ]:
n_rolls = 2000
rolls = rng.integers(1, 7, size=n_rolls)
retained = rolls[rolls >= 5]
retained_count = retained.size

if retained_count == 0:
    estimated_six_probability = None
else:
    estimated_six_probability = np.mean(retained == 6)

print("retained count N_A =", retained_count)
print("estimated P(X=6 | X in {5,6}) =", estimated_six_probability)
print("theoretical conditional probability =", 0.5)


## Questions to ask when building a model

Before computing with joint or conditional data, state:

1. the random variables and their possible values;
2. the joint PMF, joint density, or an equivalent generator/conditional law;
3. any independence assumptions;
4. which conditioning values have positive marginal probability or density;
5. whether a filter was fixed before observing the data.

Association in a joint distribution is not, by itself, evidence of a causal
relationship.


## Try it yourself

1. Using the finite table above, compute $p_{X\mid Y}(x\mid1)$ and
   $\mathbb E[X\mid Y=1]$. Check the tower property by conditioning on
   $Y$ instead of $X$.
2. Verify by integration that the triangular density integrates to one and
   derive $f_X$ and $f_Y$.
3. Change the fixed die filter to $A=\{4,5,6\}$. State the conditional
   law before simulating it, and include an explicit $N_A=0$ branch.
